# Whole U.S. — Census Tract Matched Records Export

**Goal:** Produce a clean dataset of CoStar properties that resolved to a single U.S. Census tract.
**Input:** `data/1_derived/5_whole_census_tract_matcher_001-007.csv` (post-tract-matcher batches)
**Output:** `data/1_derived/6_whole_tract_matched_final.csv`
**Filter:** `tract_resolved == True` (rows whose `final_census_tract` is non-null).

A row is `tract_resolved` if it was either a One-to-One geocoder match with a known tract, or a One-to-Many tie that the tract matcher collapsed to a single tract (via ZIP-matching ties or all-ties unanimous).

In [1]:
import os
import glob
import pandas as pd

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 20)

In [2]:
# --- Load all tract matcher output files ---
IN_DIR = os.path.join('..', '..', 'data', '1_derived')
files = sorted(glob.glob(os.path.join(IN_DIR, '5_whole_census_tract_matcher_0*.csv')))

print(f'Reading {len(files)} files...')
df = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)
print(f'Total rows loaded: {len(df):,}')
print('\nResolution method counts:')
print(df['tract_resolution_method'].value_counts(dropna=False))
print('\nResolved counts:')
print(df['tract_resolved'].value_counts(dropna=False))

Reading 7 files...


Total rows loaded: 1,051,219

Resolution method counts:
tract_resolution_method
One-to-One Primary                     1043837
Tie Resolved via ZIP-Matching Ties        3746
Tract Ambiguous                           2637
Tract Missing                              685
Tie Resolved via All Ties Unanimous        314
Name: count, dtype: int64

Resolved counts:
tract_resolved
True     1047897
False       3322
Name: count, dtype: int64


In [3]:
# --- Filter to rows that resolved to a single census tract ---
matched = df[df['tract_resolved'] == True].copy()
print(f'Tract-resolved rows: {len(matched):,} of {len(df):,} ({len(matched)/len(df)*100:.2f}%)')

Tract-resolved rows: 1,047,897 of 1,051,219 (99.68%)


In [4]:
# --- Select CoStar address fields + Census geocoding results + tract resolution metadata ---
KEEP_COLS = [
    # CoStar identifiers
    'LeaseCompId',
    'PropertyId',
    'PropertyName',
    'Market',
    'Submarket',
    # CoStar address fields
    'Address.DeliveryAddress',
    'Suite',
    'Address.Locality',
    'Address.County',
    'Address.PostalCode',
    'Address.RegionName',
    'Address.Subdivision',
    'Address.Country',
    # Census geocoding results
    'matched_address',
    'latitude',
    'longitude',
    'state_fips',
    'county_fips',
    'census_tract',
    'census_block',
    'geoid',
    # Match + tract resolution metadata
    'match',
    'match_type',
    'zip_match_detail',
    'tract_resolution_method',
    'final_census_tract',
]

# Keep only columns that exist in the dataframe
cols = [c for c in KEEP_COLS if c in matched.columns]
out = matched[cols].reset_index(drop=True)
print(f'Output shape: {out.shape}')
print('\nResolution method breakdown of exported rows:')
print(out['tract_resolution_method'].value_counts(dropna=False))
out.head(3)

Output shape: (1047897, 26)

Resolution method breakdown of exported rows:
tract_resolution_method
One-to-One Primary                     1043837
Tie Resolved via ZIP-Matching Ties        3746
Tie Resolved via All Ties Unanimous        314
Name: count, dtype: int64


,LeaseCompId,PropertyId,PropertyName,Market,Submarket,Address.DeliveryAddress,Suite,Address.Locality,Address.County,Address.PostalCode,...,state_fips,county_fips,census_tract,census_block,geoid,match,match_type,zip_match_detail,tract_resolution_method,final_census_tract
0,114028931.0,435558.0,Bldg A,Atlanta,Cumberland/Galleria,4501 Circle 75 Pky SE,1170,Atlanta,Cobb,30339.0,...,0.0,0.0,0.0,0.0,0.000000e+00,No_Match,One-to-One,Geocoder ZIP Missing,One-to-One Primary,0.0
1,113952192.0,445018.0,The Palisades Medical Office,Atlanta,Upper Buckhead,3200 Downwood Cir NW,350,Atlanta,Fulton,30327.0,...,13.0,121.0,9803.0,2002.0,1.312101e+14,Match,One-to-One,One-to-One Matched,One-to-One Primary,9803.0
2,114008333.0,440921.0,100 Edgewood,Atlanta,Downtown Atlanta,100 Edgewood Ave NE,540,Atlanta,Fulton,30303.0,...,13.0,121.0,11901.0,3012.0,1.312101e+14,Match,One-to-One,One-to-One Matched,One-to-One Primary,11901.0


In [5]:
# --- Write output (split if needed for GitHub size cap) ---
GITHUB_MAX_MB = 95
OUT_PREFIX = os.path.join(IN_DIR, '6_whole_tract_matched_final')
OUT_SINGLE = OUT_PREFIX + '.csv'
MANIFEST_PATH = OUT_PREFIX + '_manifest.csv'

max_bytes = int(GITHUB_MAX_MB * 1024 * 1024)
out.to_csv(OUT_SINGLE, index=False)
single_size = os.path.getsize(OUT_SINGLE)

if single_size <= max_bytes:
    manifest = pd.DataFrame([{
        'part_file': os.path.basename(OUT_SINGLE),
        'rows': len(out),
        'size_mb': round(single_size / (1024 * 1024), 2),
    }])
    print(f'Saved single file ({single_size/(1024*1024):.2f} MB):')
    print(f'  - {OUT_SINGLE}')
else:
    os.remove(OUT_SINGLE)
    sample_n = min(20000, len(out))
    sample_csv_bytes = len(out.head(sample_n).to_csv(index=False).encode('utf-8'))
    bytes_per_row = max(1.0, sample_csv_bytes / max(1, sample_n))
    rows_per_file = max(1, int((max_bytes / bytes_per_row) * 0.9))

    manifest_rows = []
    part = 1
    start = 0
    while start < len(out):
        end = min(start + rows_per_file, len(out))
        path = f'{OUT_PREFIX}_{part:03d}.csv'
        chunk = out.iloc[start:end]
        chunk.to_csv(path, index=False)
        size_bytes = os.path.getsize(path)
        if size_bytes > max_bytes and len(chunk) > 1:
            os.remove(path)
            rows_per_file = max(1, int((end - start) * 0.85))
            continue
        manifest_rows.append({
            'part_file': os.path.basename(path),
            'rows': len(chunk),
            'size_mb': round(size_bytes / (1024 * 1024), 2),
        })
        start = end
        part += 1
    manifest = pd.DataFrame(manifest_rows)
    print('Saved split files:')
    for r in manifest.itertuples(index=False):
        print(f'  - {r.part_file} ({r.size_mb} MB)')

manifest.to_csv(MANIFEST_PATH, index=False)
print(f'\nManifest: {MANIFEST_PATH}')
print(f"Total rows saved: {manifest['rows'].sum():,}")

Saved split files:
  - 6_whole_tract_matched_final_001.csv (85.44 MB)
  - 6_whole_tract_matched_final_002.csv (85.81 MB)
  - 6_whole_tract_matched_final_003.csv (85.44 MB)
  - 6_whole_tract_matched_final_004.csv (39.83 MB)

Manifest: ..\..\data\1_derived\6_whole_tract_matched_final_manifest.csv
Total rows saved: 1,047,897
